In [ ]:
import os
import sys
sys.path.append(os.path.dirname(os.getcwd()))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm
from matplotlib.colors import Normalize
from PIL import Image

from pc import PS
from modules import ADC,DAC,CHIP,SELECT
from command import CMD,CmdData,Packet
from command.singleCmdInfo import *

from util import *

from network.layer import Layer
from scipy.optimize import curve_fit 
import random
import time

In [ ]:
# chip=CHIP(PS(host="192.168.1.10", port = 7, debug=0),init=True)
# chip.set_device_cfg(deviceType=0,IsNew32=False)
# chip.adc.set_gap(adc_cs_gap=90,adc_first_gap=20,adc_last_gap=10)
# chip.adc.set_gain_resistor(big_resistance=10e3,small_resistance=200)
# chip.clk_manager.set_cyc(10, 10,delay3=50)
# chip.add_compiler("../compiler/code/")

In [ ]:
chip=CHIP(PS(host="192.168.1.11", port = 7, debug=0),init=True)
# deviceType参数：0为ReRAM，1为ECRAM
# IsNew32参数：False为v1版本，True为v2版本
chip.set_device_cfg(deviceType=0,IsNew32=True)
chip.adc.set_sample_times(adc_sample_times=16)
chip.adc.set_gap(adc_cs_gap=100,adc_first_gap=1000,adc_last_gap=40)
chip.adc.set_gain_resistor(big_resistance=10e3,small_resistance=200)
chip.clk_manager.set_cyc(10, 10,delay3=0)
chip.add_compiler("./compiler/code/")
# chip.compensation.initop("../chip_data/chip6_/")

In [ ]:
first_gap = 200
chip.adc.set_op_adc(data=0b1_001_00_1_001_00_0000)
samples = 1
chip.adc.set_sample_times(adc_sample_times=samples)
chip.adc.set_gap(adc_cs_gap=100,adc_first_gap=first_gap,adc_last_gap=40)

crossbar = np.ones((40,40))

for i in range(1):
    v0,c0,r = chip.read4(crossbar=crossbar,row_index=None,col_index=None,read_voltage=0.1,tg=5,gain=1,sub_base=True,from_row=False,split_type=0,row_type=0,col_type=0)
    plot_cond(v0[:40,:40],vmin=np.min(v0),vmax=np.max(v0))


rows = [i for i in range(40)]
cols = [i for i in range(40)]
pos = np.ix_(rows,cols)
for i in range(1):
    v,c,r = chip.read4(crossbar=None,row_index=rows,col_index=cols,read_voltage=0.1,tg=5,gain=1,sub_base=False,from_row=False,split_type=6,row_type=0,col_type=0)
    plot_cond(c[:40,:40],vmin=np.min(c),vmax=np.max(c))


cc = v0[:40,:40]-v[:40,:40]
plot_cond(cc,vmin=np.min(cc),vmax=np.max(cc))
plt.hist(cc)
plt.show()

In [ ]:
def adc_plot(mean_data,std_data,mean_min,mean_max,std_min,std_max,title1,title2,label):
    plt.figure(figsize=(12,4))
    # 均值
    plt.subplot(1,2,1)
    cmap = plt.cm.viridis
    norm = Normalize(vmin=mean_min, vmax=mean_max)
    im = plt.imshow(mean_data, cmap=cmap,norm=norm)
    cbar = plt.colorbar(im)
    cbar.set_label(label)
    plt.title(title1+f"-mean_{label}")
    # 标准差
    plt.subplot(1,2,2)
    cmap = plt.cm.viridis
    norm = Normalize(vmin=std_min, vmax=std_max)
    im = plt.imshow(std_data, cmap=cmap,norm=norm)
    cbar = plt.colorbar(im)
    cbar.set_label(label)
    plt.title(title2+f"-std_{label}-{np.mean(std_data):.2f}")
    # # 百分比
    # plt.subplot(1,3,3)
    # cmap = plt.cm.viridis
    # norm = Normalize(vmin=0, vmax=5)
    # im = plt.imshow(std_data/mean_data*100, cmap=cmap,norm=norm)
    # cbar = plt.colorbar(im)
    # cbar.set_label("100%")
    # plt.title(f"gian={gain}-percent")
    plt.show()

# 1. 读一块区域

In [ ]:
# chip.adc.set_op_adc(data=0b1_001_00_1_001_00_0000)
chip.adc.set_spi_div(adc_spi_div=10)

In [ ]:
rows = [i for i in range(40)]
cols = [i for i in range(40)]
pos = np.ix_(rows,cols)

# first_gap_list = [100,200,300,500,1000,1500,2000,3000,4000]
first_gap_list = [1000]
samples_list = [1,2,4,8,16,32]
# ,0b001_00_0000,0b010_00_0000,0b011_00_0000
over_samples_list = [0b000_00_0000,0b001_00_0000,0b010_00_0000]
gain_list = [1]
for over_samples in over_samples_list:
    for first_gap in first_gap_list:
        for samples in samples_list:
            for gain in gain_list:
                chip.adc.set_op_adc(data=0b1_001_00_0_000_00_0000|over_samples)
                # chip.adc.set_op_adc(data=0b1_001_00_1_000_00_0000|over_samples)
                chip.adc.set_sample_times(adc_sample_times=samples)
                chip.adc.set_gap(adc_cs_gap=400,adc_first_gap=first_gap,adc_last_gap=40)
                
                chip.adc.set_op_adc(data=0b1_001_00_0_000_00_0000|over_samples)
                # chip.adc.set_op_adc(data=0b1_001_00_1_000_00_0000|over_samples)
                chip.adc.set_sample_times(adc_sample_times=samples)
                chip.adc.set_gap(adc_cs_gap=400,adc_first_gap=first_gap,adc_last_gap=40)

                print(f"over_samples={over_samples/0b1_00_0000}_first_gap={first_gap}_samples={samples}_gain={gain}")
                ans_v = []
                ans_c = []
                for i in range(100):
                    v,c,r = chip.read4(crossbar=None,row_index=rows,col_index=cols,read_voltage=0.1,tg=5,gain=gain,sub_base=True,from_row=False,split_type=6,row_type=0,col_type=0)
                    ans_v.append(v[pos])
                    ans_c.append(c[pos])
                np.save(f"../data/adc/pcb203/normal_samples/from_row={False}_cs_gap=4_spi=10_over_samples={over_samples}_first_gap={first_gap}_samples={samples}_gain={gain}_v=0_1_voltage.npy",np.array(ans_v))
                np.save(f"../data/adc/pcb203/normal_samples/from_row={False}_cs_gap=4_spi=10_over_samples={over_samples}_first_gap={first_gap}_samples={samples}_gain={gain}_v=0_1_cond.npy",np.array(ans_c))

In [ ]:
"""
# from_row = True
normal_samples/cs_gap=8_spi=10          # 表示cs宽度8us,采用频率100/10=10MHz,测试过采样
normal_samples/cs_gap=4_spi=10          # 表示cs宽度1us,采用频率100/10=10MHz,测试过采样
normal_samples/spi=10                   # 表示cs宽度1us,采用频率100/10=10MHz,测试过采样
normal_samples/from_row=True_cs_gap=1_spi=2         # 测试从行读,不同采样次数随firstgap的变化


# from_row = False
normal_samples/cs_gap=1_spi=2           # 测试从行读,不同采样次数随firstgap的变化

normal_samples/from_row={False}_cs_gap=4_spi=10     # 测试过采样
"""


In [ ]:
# first_gap_list = [100,200,300,500,1000,1500,2000,3000,4000]
# samples_list = [1,2,4,8,16,32]
# over_samples_list = [0b001,0b010,0b011]



first_gap_list = [1000]
samples_list = [1,2,4,8,16,32]
over_samples_list = [0b000_00_0000,0b001_00_0000,0b010_00_0000]
gain_list = [1]
ans_delay = []

for samples in samples_list:
    ans_delay.append([])
    for k,over_samples in enumerate(over_samples_list):
        for first_gap in first_gap_list:
            
            for gain in gain_list:
                path_v = f"../data/adc/pcb203/normal_samples/from_row={False}_cs_gap=4_spi=10_over_samples={over_samples}_first_gap={first_gap}_samples={samples}_gain={gain}_v=0_1_voltage.npy"
                path_c = f"../data/adc/pcb203/normal_samples/from_row={False}_cs_gap=4_spi=10_over_samples={over_samples}_first_gap={first_gap}_samples={samples}_gain={gain}_v=0_1_cond.npy"
                # ans_v = np.load(f"../data/adc/pcb203/first_gap={first_gap}_samples={samples}_gain={gain}_v=0_1_voltage.npy")
                # ans_c = np.load(f"../data/adc/pcb203/first_gap={first_gap}_samples={samples}_gain={gain}_v=0_1_cond.npy")
                ans_v = np.load(path_v)
                ans_c = np.load(path_c)
                # print(ans_c)
                # matrices = np.array(ans_v)
                # mean_matrix = np.mean(matrices, axis=0)
                # var_matrix = np.var(matrices, axis=0)
                # std_matrix = np.sqrt(var_matrix)
                # adc_plot(mean_data=mean_matrix*1000,std_data=std_matrix*1000,mean_max=20,std_max=0.2,gain=gain,samples=samples,label="mv")


                matrices = np.array(ans_c)
                mean_matrix = np.mean(matrices, axis=0)
                std_matrix = np.std(matrices,axis=0)
                ans_delay[-1].append(np.mean(std_matrix))
                std_max = 10 if samples>=8 else 20
                print(f"over_samples={k*2}_first_gap={first_gap}_samples={samples}_gain={gain}")
                adc_plot(mean_data=mean_matrix,std_data=std_matrix,mean_min=0,mean_max=1200,std_min=0,std_max=std_max,
                         title1=f"over_samples={k*2}_first_gap={first_gap}\n_samples={samples}_gain={gain}",
                         title2=f"over_samples={k*2}_first_gap={first_gap}\n_samples={samples}_gain={gain}",label="us")


for i,data in enumerate(ans_delay):
    plt.plot([1,2,4],data,label=f"samples={samples_list[i]}")
plt.ylabel("std cond(uS)")
plt.xlabel("over samples times")
plt.legend()
plt.show()

In [ ]:
# first_gap_list = [100,200,300,500,1000,1500,2000,3000,4000]
# samples_list = [1,2,4,8,16,32]
# over_samples_list = [0b001,0b010,0b011]



first_gap_list =  [100,200,300,500,1000,1500,2000,3000,4000]
samples_list = [1,2,4,8,16,32]
gain_list = [1]
ans_delay = []
for samples in samples_list:
    ans_delay.append([])
    for first_gap in first_gap_list:
        
        for gain in gain_list:
            path_v = f"../data/adc/pcb203/normal_samples/cs_gap=1_spi=2_over_samples={over_samples}_first_gap={first_gap}_samples={samples}_gain={gain}_v=0_1_voltage.npy"
            path_c = f"../data/adc/pcb203/normal_samples/cs_gap=1_spi=2_over_samples={over_samples}_first_gap={first_gap}_samples={samples}_gain={gain}_v=0_1_cond.npy"
            # ans_v = np.load(f"../data/adc/pcb203/first_gap={first_gap}_samples={samples}_gain={gain}_v=0_1_voltage.npy")
            # ans_c = np.load(f"../data/adc/pcb203/first_gap={first_gap}_samples={samples}_gain={gain}_v=0_1_cond.npy")
            ans_v = np.load(path_v)
            ans_c = np.load(path_c)
            # print(ans_c)
            # matrices = np.array(ans_v)
            # mean_matrix = np.mean(matrices, axis=0)
            # var_matrix = np.var(matrices, axis=0)
            # std_matrix = np.sqrt(var_matrix)
            # adc_plot(mean_data=mean_matrix*1000,std_data=std_matrix*1000,mean_max=20,std_max=0.2,gain=gain,samples=samples,label="mv")


            matrices = np.array(ans_c)
            mean_matrix = np.mean(matrices, axis=0)
            std_matrix = np.std(matrices,axis=0)
            ans_delay[-1].append(np.mean(std_matrix))
            std_max = 10 if samples>=8 else 20
            adc_plot(mean_data=mean_matrix,std_data=std_matrix,mean_min=0,mean_max=1200,std_min=0,std_max=std_max,
                        title1=f"over_samples={0}_first_gap={first_gap}\n_samples={samples}_gain={gain}",
                        title2=f"over_samples={0}_first_gap={first_gap}\n_samples={samples}_gain={gain}",label="us")
for i,data in enumerate(ans_delay):
    plt.plot(np.array(first_gap_list)/100,data,label=f"samples={samples_list[i]}")
plt.ylabel("std cond(uS)")
plt.xlabel("cs_first_gap(us)")
plt.legend()
plt.show()

In [ ]:
first_gap = 200
samples = 1
gain = 1            
ans_v = np.load(f"../data/adc/pcb203/first_gap={first_gap}_samples={samples}_gain={gain}_v=0_1_voltage.npy")
ans_c = np.load(f"../data/adc/pcb203/first_gap={first_gap}_samples={samples}_gain={gain}_v=0_1_cond.npy")

for i in range(100):
    plot_cond(ans_c[i])
matrices = np.array(ans_c)
mean_matrix = np.mean(matrices, axis=0)
std_matrix = np.std(matrices,axis=0)
ans_delay[-1].append(np.mean(std_matrix))
std_max = 10 if samples>=8 else 20
adc_plot(mean_data=mean_matrix,std_data=std_matrix,mean_max=1200,std_max=std_max,gain=gain,samples=samples,first_gap=(first_gap/100),label="us")

In [ ]:

first_gap = 100
samples = 32
gain = 1
ans_c = np.load(f"../data/adc/pcb203/first_gap={first_gap}_samples={samples}_gain={gain}_v=0_1_cond.npy")
for row in range(2):
    for col in range(5):
        plt.plot(ans_c[:,row,col])
        plt.title(f"row = {row},col={col},std={np.std(ans_c[:,row,col]):.4f}")
        plt.show()

# 2. 读一个点

In [ ]:
row =100# random.randint(0,256)
col = 100#random.randint(0,256)
ans = []

first_gap = 200
samples = 32
gain = 0
from_row= True

chip.adc.set_sample_times(adc_sample_times=samples)
chip.adc.set_gap(adc_cs_gap=100,adc_first_gap=first_gap,adc_last_gap=40)
for i in range(1):
    # time.sleep(4)
    for j in range(100):
        v,c,r = chip.read4(crossbar=None,row_index=[row],col_index=[col],read_voltage=0.2,tg=5,gain=gain,sub_base=True,from_row=from_row,split_type=4,row_type=0,col_type=0)
        ans.append(c[row])

In [ ]:
print(v[100])
print(chip.setting.TIA_index_map(num=100,col=False))

In [ ]:
plt.plot(ans)
plt.title(f"row={row},col={col},from_row={from_row}")
plt.show()

In [ ]:
crossbar = np.ones((256,256))
v,c,r = chip.read4(crossbar=crossbar,row_index=None,col_index=None,read_voltage=0.1,tg=5,gain=1,sub_base=True,from_row=False,split_type=1,row_type=0,col_type=0)
plot_cond(c,vmax=1000)

In [ ]:
crossbar = np.zeros((256,256))
ans = []
ans1 = []
row =11
col =100
adc_first_gap = 200
chip.adc.set_gap(adc_cs_gap=100,adc_first_gap=adc_first_gap,adc_last_gap=30)
chip.adc.set_sample_times(adc_sample_times=32)

crossbar[row,col]=1
sleep_time = 5
gain = 1
for k in range(5):
    # time.sleep(sleep_time)
    for i in range(100):
        v,cond,_ = chip.read4(crossbar=crossbar,row_index=[row],col_index=[col],read_voltage=0.1,tg=5,gain=gain,sub_base=True,from_row=False,split_type=0,row_type=0,col_type=0)
        ans.append(cond[row,col])
        ans1.append(v[row,col])

plt.plot(ans[:])
plt.ylabel("uS")
plt.xlabel("read times")
plt.title(f"row={row},col={col},gain = {gain},sleep time = {sleep_time}s,adc_first_gap={adc_first_gap*10}ns")
plt.show()

In [ ]:
plt.plot(ans)
plt.show()

In [ ]:
for first_gap in [100,200,500,1000]:
    for samples in [1,2,4,8,16,32]:
        # time.sleep(4)
        chip.adc.set_sample_times(adc_sample_times=samples)
        chip.adc.set_gap(adc_cs_gap=100,adc_first_gap=first_gap,adc_last_gap=40)
        ans = []
        row = 10
        col = 100
        for i in range(100):
            v,c,r = chip.read4(crossbar=None,row_index=[row],col_index=[col],read_voltage=0.1,tg=5,gain=1,sub_base=True,from_row=False,split_type=6,row_type=0,col_type=0)
            ans.append(c[row,col])
        # var_matrix = np.var(matrices, axis=0)
        std_matrix = np.std(ans)
        plt.plot(ans)
        plt.ylabel("us")
        plt.xlabel("random times")
        plt.title(f"first_gap = {first_gap},samples={samples},std={std_matrix:.4f}")
        plt.show()

In [ ]:
# chip.setting.ins_ram_length = 

In [ ]:
crossbar = np.ones((256,256))
v,c,r = chip.read4(crossbar=crossbar,row_index=None,col_index=None,read_voltage=0.1,tg=5,gain=1,sub_base=True,from_row=False,split_type=1,row_type=0,col_type=0)
plot_cond(c,vmax=1000)

In [ ]:

for col in range(1,256):
    ans = []
    ans2 = []
    for i in range(50):
        num = random.randint(5, 15)
        start = random.randint(10, 200)
        v,c,r = chip.read4(crossbar=None,row_index=[i for i in range(start,start+num)],col_index=[col],read_voltage=0.1,tg=5,gain=1,sub_base=True,from_row=True,split_type=3,row_type=0,col_type=0)
        ans.append(c[col])
        v,c,r = chip.read4(crossbar=None,row_index=[i for i in range(start,start+num)],col_index=[col],read_voltage=0.1,tg=5,gain=3,sub_base=True,from_row=True,split_type=3,row_type=0,col_type=0)
        ans2.append(c[col])

    params, covariance = curve_fit(lambda x, mult: x * mult, np.array(ans2),np.array(ans), p0=[0.94],bounds=([0], [2]))
    print(params[0])

# np.save(root_path + "row_gain_r_mult.npy",gain_r_mult_fit)

In [ ]:
ans = np.array(ans)
ans2 = np.array(ans2)
plt.plot(ans,label="gain1")
plt.plot(ans2*0.93801281,label="gain3")
plt.legend()
plt.show()

# plt.plot(ans,label="gain1")
plt.plot(np.array(ans)/np.array(ans2),label="gain1/gain3")
plt.ylim((0.93,0.95))
plt.legend()
plt.show()

In [ ]:
ans = np.array(ans)
ans2 = np.array(ans2)
plt.plot(ans,label="gain1")
plt.plot(ans2*0.93801281,label="gain3")
plt.legend()
plt.show()

# plt.plot(ans,label="gain1")
plt.plot(np.array(ans)/np.array(ans2),label="gain1/gain3")
plt.ylim((0.93,0.95))
plt.legend()
plt.show()

In [ ]:

crossbar = np.ones((50,50))
for gain in [3,1,2,0]:
    ans = np.zeros((50,50,50))
    ansv = np.zeros((50,50,50))
    for i in range(50):
        v,c,r = chip.read4(crossbar=crossbar,row_index=None,col_index=None,read_voltage=0.1,tg=5,gain=gain,sub_base=True,from_row=False,split_type=0,row_type=0,col_type=0)
        ans[i,:,:]=c[:50,:50]
        ansv[i,:,:]=v[:50,:50]
    np.save(f"./data/adc/pcb203/gain/gain={gain}_first_gap={2}_from_row={False}_sub_base={True}_c.npy",ans)
    np.save(f"./data/adc/pcb203/gain/gain={gain}_first_gap={2}_from_row={False}_sub_base={True}_v.npy",ansv)

In [ ]:
gain = 3
ans_3 = np.load(f"../data/adc/pcb203/gain/gain={gain}_first_gap={2}_from_row={False}_sub_base={True}_c.npy")
ansv_3 = np.load(f"../data/adc/pcb203/gain/gain={gain}_first_gap={2}_from_row={False}_sub_base={True}_v.npy")


gain = 1
ans_1 = np.load(f"../data/adc/pcb203/gain/gain={gain}_first_gap={2}_from_row={False}_sub_base={True}_c.npy")
ansv_1 = np.load(f"../data/adc/pcb203/gain/gain={gain}_first_gap={2}_from_row={False}_sub_base={True}_v.npy")


gain = 2
ans_2 = np.load(f"../data/adc/pcb203/gain/gain={gain}_first_gap={2}_from_row={False}_sub_base={True}_c.npy")
ansv_2 = np.load(f"../data/adc/pcb203/gain/gain={gain}_first_gap={2}_from_row={False}_sub_base={True}_v.npy")


gain = 0
ans_0 = np.load(f"../data/adc/pcb203/gain/gain={gain}_first_gap={2}_from_row={False}_sub_base={True}_c.npy")
ansv_0 = np.load(f"../data/adc/pcb203/gain/gain={gain}_first_gap={2}_from_row={False}_sub_base={True}_v.npy")


# mean_matrix = np.mean(matrices, axis=0)[:20,:180]
ans = [np.mean(ans_0, axis=0),np.mean(ans_1, axis=0),np.mean(ans_2, axis=0),np.mean(ans_3, axis=0)]

In [ ]:
for i in range(1,2):
    plot_cond((ans[3]-ans[i])/ans[3],vmax = 2/15)
    plot_cond(ans[i])
    plt.hist((ans[3]-ans[1])/ans[i])
    # plt.title(f"gian={i}_read")
    plt.show()
    # plt.hist((ans[3]-ans[i])/ans[3])
    # plt.title(f"gian={3} - gain={i}")
    # plt.show()

In [ ]:
for i in range(4):
    plt.hist(ans[i])
    plt.title(f"gian={i}_read")
    plt.show()
    plt.hist((ans[3]-ans[i])/ans[3])
    plt.title(f"gian={3} - gain={i}")
    plt.show()
    
# plt.hist(ans[1]-ans[3])
# plt.show()
# plot_cond(ans[3]-ans[1],vmax=200)
# plot_cond(ans[1])